<a href="https://colab.research.google.com/github/ravindravala/AI/blob/main/test_CNN_with_MNIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader,Dataset
import torchvision
import torchvision.transforms as transforms

In [3]:
transform = transforms.Compose([transforms.ToTensor()])

In [4]:
train_dataset = torchvision.datasets.MNIST(root='./data',train=True,transform=transform,download=True)
test_dataset = torchvision.datasets.MNIST(root='./data',train=False,transform=transform,download=True)


100%|██████████| 9.91M/9.91M [00:01<00:00, 5.10MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 134kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.27MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.44MB/s]


In [5]:
train_loader = DataLoader(dataset=train_dataset,batch_size=100,shuffle=True)
test_loader = DataLoader(dataset=test_dataset,batch_size=100,shuffle=False)

In [7]:
image,label = next(iter(train_loader))
print(image.size())

torch.Size([100, 1, 28, 28])


In [9]:
class SimpleCNN(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(1,16,3)
    self.pool = nn.MaxPool2d(2,2)
    self.linear = nn.Linear(16 * 13 * 13, 10)

  def forward(self,x):
    x = self.pool(F.relu(self.conv1(x)))
    x = x.view(x.size(0),-1)
    x = self.linear(x)
    return x

In [10]:
model = SimpleCNN()
optimizer = optim.Adam(model.parameters(),lr=0.001)
criterion = nn.CrossEntropyLoss()

In [11]:
epochs=5

for epoch in range(epochs):
  for x,y in train_loader:
    pred = model(x)
    loss = criterion(pred,y)

    model.zero_grad()
    loss.backward()
    optimizer.step()

  print(f"Epoch: {epoch+1}, Loss: {loss.item()}")

Epoch: 1, Loss: 0.22984947264194489
Epoch: 2, Loss: 0.09231476485729218
Epoch: 3, Loss: 0.04054617881774902
Epoch: 4, Loss: 0.08208112418651581
Epoch: 5, Loss: 0.09519986063241959


In [13]:
model.eval()
with torch.no_grad():
  correct=0
  total=0
  for x,y in test_loader:
    pred = model(x)
    _,pred_label = torch.max(pred,1)
    correct += (pred_label == y).sum().item()
    total += y.size(0)

  print(f"Accuracy: {100 * (correct/total)}")


Accuracy: 97.89
